## 표정 유사도 채점 (웃음벨 - 표정 모듈)

기획서 4.2의 정석: '생김새'가 아니라 '표정의 모양(blendshape 52계수)'을 비교한다.

**채점 방식 (코사인의 약점 보완):**
- 방향 점수(cosine): 어떤 *종류*의 표정인가
- 모양·세기 점수(가중 오차): 같은 근육을 같은 *세기*로 썼나  ← 코사인이 못 잡는 강도까지 평가
- 최종 = 방향 0.4 + 모양·세기 0.6

**사용법:** `models/face_landmarker.task` 와 `data/` 이미지를 준비한 뒤 셀을 위에서부터 실행.
모델은 https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task


In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


def imread_unicode(path):
    """cv2.imread는 윈도우 한글 경로에서 실패 -> np.fromfile + imdecode로 우회."""
    data = np.fromfile(path, dtype=np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)


def build_detector(model_path="models/face_landmarker.task"):
    """blendshape(표정 계수 52개) 출력을 켠 Face Landmarker 생성."""
    base = python.BaseOptions(model_asset_path=model_path)
    opts = vision.FaceLandmarkerOptions(
        base_options=base, output_face_blendshapes=True, num_faces=1
    )
    return vision.FaceLandmarker.create_from_options(opts)


def extract_blendshapes(detector, image_path):
    """이미지 1장 -> 52차원 표정계수 벡터(이름순 정렬), {이름:값} 딕셔너리. 실패 시 (None, None)."""
    bgr = imread_unicode(image_path)
    if bgr is None:
        return None, None
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    res = detector.detect(image)
    if not res.face_blendshapes:
        return None, None
    shapes = sorted(res.face_blendshapes[0], key=lambda c: c.category_name)
    names = [c.category_name for c in shapes]
    vec = np.array([c.score for c in shapes], dtype=np.float32)
    return vec, dict(zip(names, vec))


def top_expressions(shape_dict, k=5):
    """가장 강하게 활성화된 표정 계수 (사람 눈으로 점수 검증용)."""
    items = sorted(shape_dict.items(), key=lambda kv: kv[1], reverse=True)
    return [(n, round(float(v), 2)) for n, v in items[:k] if v > 0.05]


def score_expression(target, user, k=180.0, w_dir=0.4, w_shape=0.6):
    """표정 유사도 0~100.
    - cosine(방향): 표정의 '종류'
    - shape(가중 평균절대오차): 같은 근육을 같은 '세기'로 썼나. 가중치=max(target,user)로
      활성화된 근육만 평가 -> 무표정 성분에 의한 점수 인플레 방지.
    """
    cos = float(np.dot(target, user) / (np.linalg.norm(target) * np.linalg.norm(user) + 1e-8))
    cos_score = max(0.0, cos) * 100

    w = np.maximum(target, user)
    werr = float(np.sum(w * np.abs(target - user)) / (np.sum(w) + 1e-8))
    shape_score = max(0.0, 100 - werr * k)

    final = w_dir * cos_score + w_shape * shape_score
    return {"final": round(final, 1), "cosine": round(cos_score, 1),
            "shape": round(shape_score, 1), "werr": round(werr, 3)}


def evaluate_expression_similarity(target_path, user_path, detector, k=180.0):
    """팀원 evaluate_pose_similarity와 동일한 (점수, 메시지) 반환 -> 나중에 표정+동작 합산 쉬움."""
    tv, td = extract_blendshapes(detector, target_path)
    uv, ud = extract_blendshapes(detector, user_path)
    if tv is None:
        return 0.0, f"인식 실패(목표): {target_path}"
    if uv is None:
        return 0.0, f"인식 실패(유저): {user_path}"
    s = score_expression(tv, uv, k=k)
    msg = (f"성공 (방향 {s['cosine']} / 모양·세기 {s['shape']})  "
           f"목표:{top_expressions(td)}  유저:{top_expressions(ud)}")
    return s["final"], msg

In [ ]:
# 모델 로드 (이 셀은 한 번만 실행하면 됨. 아래 테스트 셀만 반복 실행하며 튜닝)
detector = build_detector("models/face_landmarker.task")
print("표정 채점기 준비 완료")

In [ ]:
import os, glob

# ── 여기를 바꿔가며 반복 실행하면서 튜닝하세요 ──────────────
K = 180.0                       # 클수록 엄격(점수 차이 커짐). 분별력 보고 조절
target_img = "data/origin.jpg"  # 따라 할 목표 표정
test_dir   = "data/test_images" # 채점할 유저 사진들이 든 폴더
# ───────────────────────────────────────────────

exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
files = sorted(p for p in glob.glob(os.path.join(test_dir, "*"))
               if p.lower().endswith(exts))

results = []
for p in files:
    score, msg = evaluate_expression_similarity(target_img, p, detector, k=K)
    results.append((os.path.basename(p), score, msg))

results.sort(key=lambda r: r[1], reverse=True)  # 높은 점수 순

print(f"목표 표정: {target_img}\n")
print(f"{'순위':<5}{'점수':>8}   파일명")
print("-" * 40)
for rank, (name, score, msg) in enumerate(results, 1):
    print(f"{rank:<5}{score:>8.2f}   {name}")

if results:
    best = results[0]
    print(f"\n🔥 가장 잘 따라한 사진: {best[0]} ({best[1]:.2f}점)")
    print(f"   상세: {best[2]}")